# Notebook 2: Exploratory Data Analysis

Visual exploration of the 8 insights that `stage2_eda.py` computes on the cluster.  
Here we reproduce them locally with **pandas + matplotlib** for understanding and report preparation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

DATA_PATH = "../data/sf_incidents.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
df["Incident Datetime"] = pd.to_datetime(df["Incident Datetime"], format="%Y/%m/%d %I:%M:%S %p")
df["hour"]  = df["Incident Datetime"].dt.hour
df["month"] = df["Incident Datetime"].dt.month
df["year"]  = df["Incident Datetime"].dt.year

print(f"Loaded {len(df):,} rows")

## Insight 1 — Top 10 Incident Categories

In [ ]:
top10 = df["Incident Category"].value_counts().head(10)

fig, ax = plt.subplots()
top10.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Incident Count")
ax.set_title("Top 10 Incident Categories")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
plt.tight_layout()
plt.savefig("../output/eda/plot_insight1_top_categories.png", dpi=150)
plt.show()
top10.reset_index().rename(columns={"index": "category", "Incident Category": "count"})

## Insight 2 — Incidents by Hour of Day

In [ ]:
by_hour = df.groupby("hour").size().rename("count")

fig, ax = plt.subplots()
by_hour.plot(kind="line", ax=ax, marker="o", color="steelblue")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Incident Count")
ax.set_title("Incidents by Hour of Day")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig("../output/eda/plot_insight2_by_hour.png", dpi=150)
plt.show()

## Insight 3 — Incidents by Day of Week

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = df["Incident Day of Week"].value_counts().reindex(order)

fig, ax = plt.subplots()
by_day.plot(kind="bar", ax=ax, color="steelblue", rot=0)
ax.set_ylabel("Incident Count")
ax.set_title("Incidents by Day of Week")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
plt.tight_layout()
plt.savefig("../output/eda/plot_insight3_by_day.png", dpi=150)
plt.show()

## Insight 4 — Incidents by Police District

In [ ]:
by_district = df["Police District"].value_counts(dropna=False).fillna("Unknown")

fig, ax = plt.subplots()
by_district.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Incident Count")
ax.set_title("Incidents by Police District")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
plt.tight_layout()
plt.savefig("../output/eda/plot_insight4_by_district.png", dpi=150)
plt.show()

## Insight 5 — Monthly Trend Over Years

In [ ]:
monthly = df.groupby(["year", "month"]).size().rename("count").reset_index()
monthly["period"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))
monthly = monthly.set_index("period")["count"]

fig, ax = plt.subplots(figsize=(14, 4))
monthly.plot(ax=ax, color="steelblue")
ax.set_ylabel("Incident Count")
ax.set_title("Monthly Incident Trend (2018–Present)")
plt.tight_layout()
plt.savefig("../output/eda/plot_insight5_monthly_trend.png", dpi=150)
plt.show()

## Insight 6 — Resolution Rate by Top 10 Categories

In [ ]:
top10_cats = df["Incident Category"].value_counts().head(10).index
sub = df[df["Incident Category"].isin(top10_cats)].copy()
sub["resolved"] = sub["Resolution"] != "Open or Active"

res_rate = sub.groupby("Incident Category")["resolved"].mean().sort_values() * 100

fig, ax = plt.subplots()
res_rate.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Resolution Rate (%)")
ax.set_title("Resolution Rate by Top 10 Categories")
ax.axvline(50, color="red", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.savefig("../output/eda/plot_insight6_resolution_rate.png", dpi=150)
plt.show()
res_rate.round(1)

## Insight 7 — Top 15 Neighborhoods

In [ ]:
top_hoods = df["Analysis Neighborhood"].value_counts().head(15)

fig, ax = plt.subplots()
top_hoods.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Incident Count")
ax.set_title("Top 15 Neighborhoods by Incident Count")
plt.tight_layout()
plt.savefig("../output/eda/plot_insight7_neighborhoods.png", dpi=150)
plt.show()

## Insight 8 — Heatmap: Hour × Day of Week

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap = df.groupby(["hour", "Incident Day of Week"]).size().unstack()[order]

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(heatmap.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(7))
ax.set_xticklabels([d[:3] for d in order])
ax.set_yticks(range(24))
ax.set_yticklabels(range(24))
ax.set_xlabel("Day of Week")
ax.set_ylabel("Hour of Day")
ax.set_title("Incident Heatmap: Hour × Day of Week")
plt.colorbar(im, ax=ax, label="Incident Count")
plt.tight_layout()
plt.savefig("../output/eda/plot_insight8_heatmap.png", dpi=150)
plt.show()